In [17]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import yaml
import time
import os
import hashlib
import sys
from pathlib import Path

try:
    script_dir = Path(__file__).resolve().parent
except NameError:
    script_dir = Path.cwd()

grandparent_dir = script_dir.parent.parent
sys.path.append(str(grandparent_dir))

import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression, histogram_1D_config_creator
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [18]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/hal/ALL_PHYS_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS2_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/hal/ALL_PHYS3_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS2_SEMILEPTONIC_NoSmearing/*.root"
)
all_files += glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS3_SEMILEPTONIC_NoSmearing/*.root"
)

data_files = glob.glob(
  f"/workspace/root_files/hal/DATA_SEMILEPTONIC_NoSmearing/*.root"
  )
data_files += glob.glob(
  f"/workspace/root_files/kitt/DATA_SEMILEPTONIC_NoSmearing/*.root"
  )

# Użycie (błyskawiczne!):
all_files = get_valid_files_fast(all_files)
data_files = get_valid_files_fast(data_files)

all_files = all_files[:10]
data_files = data_files[:10]

# Scaling factor
lumi_data = 1 # 1634433.9750
lumi_mc = 1 # 1444990.0313
scaling_factor = lumi_data / lumi_mc
print(f"Lumi factor: {scaling_factor:.4f}")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_data = ROOT.RDataFrame("h1", data_files)

mPi0 = 134.9768 # MeV
mPiCh = 139.57039 # MeV
mK0 = 497.611 # MeV

# --- PRZETWARZANIE DANYCH (DATA) ---
df_data = (
    df_data
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonChTimeCMSignalFit - KaonNeTimeCMSignalFit")
    .Define("Delta_t_MC", "KaonChTimeCMMC - KaonNeTimeCMMC")
    .Define("Delta_t_rec", "KaonChTimeCMBoostLor - KaonNeTimeCMBoostLor")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[0]")
    .Define("trk2_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[1]")
    .Define("trk1_px_resolution", "trk1_matched[0] - trk1MC[0]")
    .Define("trk2_px_resolution", "trk2_matched[0] - trk2MC[0]")
    .Define("trk1_py_resolution", "trk1_matched[1] - trk1MC[1]")
    .Define("trk2_py_resolution", "trk2_matched[1] - trk2MC[1]")
    .Define("trk1_pz_resolution", "trk1_matched[2] - trk1MC[2]")
    .Define("trk2_pz_resolution", "trk2_matched[2] - trk2MC[2]")
    .Define("Kchrec_px_resolution", "Kchrec[0] - Kchmc[0]")
    .Define("Kchrec_py_resolution", "Kchrec[1] - Kchmc[1]")
    .Define("Kchrec_pz_resolution", "Kchrec[2] - Kchmc[2]")
    .Define("Kchrec_E_resolution", "Kchrec[3] - Kchmc[3]")
    .Define("Kchrec_x_resolution", "Kchrec[6] - Kchmc[6]")
    .Define("Kchrec_y_resolution", "Kchrec[7] - Kchmc[7]")
    .Define("Kchrec_z_resolution", "Kchrec[8] - Kchmc[8]")
    .Define("Kchboost_px_resolution", "Kchboost[0] - Kchmc[0]")
    .Define("Kchboost_py_resolution", "Kchboost[1] - Kchmc[1]")
    .Define("Kchboost_pz_resolution", "Kchboost[2] - Kchmc[2]")
    .Define("Kchboost_E_resolution", "Kchboost[3] - Kchmc[3]")
    .Define("Knerec_px_resolution", "Knerec[0] - Knemc[0]")
    .Define("Knerec_py_resolution", "Knerec[1] - Knemc[1]")
    .Define("Knerec_pz_resolution", "Knerec[2] - Knemc[2]")
    .Define("Knerec_E_resolution", "Knerec[3] - Knemc[3]")
    .Define("Knerec_x_resolution", "Knerec[6] - Knemc[6]")
    .Define("Knerec_y_resolution", "Knerec[7] - Knemc[7]")
    .Define("Knerec_z_resolution", "Knerec[8] - Knemc[8]")    
    .Define("KnetriKinFit_px_resolution", "KnetriKinFit[0] - Knemc[0]")
    .Define("KnetriKinFit_py_resolution", "KnetriKinFit[1] - Knemc[1]")
    .Define("KnetriKinFit_pz_resolution", "KnetriKinFit[2] - Knemc[2]")
    .Define("KnetriKinFit_E_resolution", "KnetriKinFit[3] - Knemc[3]")
    .Define("KnetriKinFit_x_resolution", "KnetriKinFit[6] - Knemc[6]")
    .Define("KnetriKinFit_y_resolution", "KnetriKinFit[7] - Knemc[7]")
    .Define("KnetriKinFit_z_resolution", "KnetriKinFit[8] - Knemc[8]")
    .Define("KchrecFit_px_resolution", "KchrecFit[0] - Kchmc[0]")
    .Define("KchrecFit_py_resolution", "KchrecFit[1] - Kchmc[1]")
    .Define("KchrecFit_pz_resolution", "KchrecFit[2] - Kchmc[2]")
    .Define("KchrecFit_E_resolution", "KchrecFit[3] - Kchmc[3]")
    .Define("KchrecFit_x_resolution", "KchrecFit[6] - Kchmc[6]")
    .Define("KchrecFit_y_resolution", "KchrecFit[7] - Kchmc[7]")
    .Define("KchrecFit_z_resolution", "KchrecFit[8] - Kchmc[8]")
    .Define("KchboostFit_px_resolution", "KchboostFit[0] - Kchmc[0]")
    .Define("KchboostFit_py_resolution", "KchboostFit[1] - Kchmc[1]")
    .Define("KchboostFit_pz_resolution", "KchboostFit[2] - Kchmc[2]")
    .Define("KchboostFit_E_resolution", "KchboostFit[3] - Kchmc[3]")
    .Define("KchboostFit_x_resolution", "KchboostFit[6] - Kchmc[6]")
    .Define("KchboostFit_y_resolution", "KchboostFit[7] - Kchmc[7]")
    .Define("KchboostFit_z_resolution", "KchboostFit[8] - Kchmc[8]")       
    .Define("KnerecFit_px_resolution", "KnerecFit[0] - Knemc[0]")
    .Define("KnerecFit_py_resolution", "KnerecFit[1] - Knemc[1]")
    .Define("KnerecFit_pz_resolution", "KnerecFit[2] - Knemc[2]")
    .Define("KnerecFit_E_resolution", "KnerecFit[3] - Knemc[3]")
    .Define("KnerecFit_x_resolution", "KnerecFit[6] - Knemc[6]")
    .Define("KnerecFit_y_resolution", "KnerecFit[7] - Knemc[7]")
    .Define("KnerecFit_z_resolution", "KnerecFit[8] - Knemc[8]")
    .Define("ip_bhabha_x_resolution", "Bx - ipmc[0]")
    .Define("ip_bhabha_y_resolution", "By - ipmc[1]")
    .Define("ip_bhabha_z_resolution", "Bz - ipmc[2]")
    .Define("ip_x_resolution", "ip[0] - ipmc[0]")
    .Define("ip_y_resolution", "ip[1] - ipmc[1]")
    .Define("ip_z_resolution", "ip[2] - ipmc[2]")
    .Define("ipFit_x_resolution", "ipFit[0] - ipmc[0]")
    .Define("ipFit_y_resolution", "ipFit[1] - ipmc[1]")
    .Define("ipFit_z_resolution", "ipFit[2] - ipmc[2]")
    .Define("KchrecPMFit_px_resolution", "KchrecPMFit[0] - Kchmc[0]")
    .Define("KchrecPMFit_py_resolution", "KchrecPMFit[1] - Kchmc[1]")
    .Define("KchrecPMFit_pz_resolution", "KchrecPMFit[2] - Kchmc[2]")
    .Define("KchrecPMFit_E_resolution", "KchrecPMFit[3] - Kchmc[3]")
    .Define("KchrecPMFit_x_resolution", "KchrecPMFit[6] - Kchmc[6]")
    .Define("KchrecPMFit_y_resolution", "KchrecPMFit[7] - Kchmc[7]")
    .Define("KchrecPMFit_z_resolution", "KchrecPMFit[8] - Kchmc[8]")
    .Define("KchboostPMFit_px_resolution", "KchboostPMFit[0] - Kchmc[0]")
    .Define("KchboostPMFit_py_resolution", "KchboostPMFit[1] - Kchmc[1]")
    .Define("KchboostPMFit_pz_resolution", "KchboostPMFit[2] - Kchmc[2]")
    .Define("KchboostPMFit_E_resolution", "KchboostPMFit[3] - Kchmc[3]")
    .Define("KchboostPMFit_x_resolution", "KchboostPMFit[6] - Kchmc[6]")
    .Define("KchboostPMFit_y_resolution", "KchboostPMFit[7] - Kchmc[7]")
    .Define("KchboostPMFit_z_resolution", "KchboostPMFit[8] - Kchmc[8]")
    .Define("KnerecNeutralFit_px_resolution", "KnerecNeutralFit[0] - Knemc[0]")
    .Define("KnerecNeutralFit_py_resolution", "KnerecNeutralFit[1] - Knemc[1]")
    .Define("KnerecNeutralFit_pz_resolution", "KnerecNeutralFit[2] - Knemc[2]")
    .Define("KnerecNeutralFit_E_resolution", "KnerecNeutralFit[3] - Knemc[3]")
    .Define("KnerecNeutralFit_x_resolution", "KnerecNeutralFit[6] - Knemc[6]")
    .Define("KnerecNeutralFit_y_resolution", "KnerecNeutralFit[7] - Knemc[7]")
    .Define("KnerecNeutralFit_z_resolution", "KnerecNeutralFit[8] - Knemc[8]")
    .Define("Kchrec_5_resolution", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5_resolution", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5_resolution", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5_resolution", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01_resolution", f"pi01[5] - {mPi0}")
    .Define("MassPi02_resolution", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit_resolution", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit_resolution", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1_resolution", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2_resolution", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1_resolution", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2_resolution", f"pi0PMFit2[5] - {mPi0}")
    ###############################################################################################
    .Filter("mcflag == 0 && mctruth != -1")
)

df_data = (
  df_data.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
      .Define("Knerec_5", f"Knerec[5] - {mK0}")
      .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
      .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
      .Define("MassPi01", f"pi01[5] - {mPi0}")
      .Define("MassPi02", f"pi02[5] - {mPi0}")
      .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
      .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
      .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
      .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
      .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
      .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
)

# --- PRZETWARZANIE MONTE CARLO (MC) ---
df = (
    df
    .Define("rho_ch", "calc_cylindrical_radius(Kchrec, ip)")
    .Define("rho_ne", "calc_cylindrical_radius(Knerec, ip)")
    .Define("z_ch", "calc_z_component(Kchrec, ip)")
    .Define("z_ne", "calc_z_component(Knerec, ip)")
    ######################### Variables for Omega-pi0 geometrical cut ############################
    .Define("bhabha_IP", "ROOT::RVec<double>{Bx, By, Bz}")
    .Define("bhabha_IP_changed", "ROOT::RVec<double>{Bx, By, KchrecClosest[8]}")
    .Define("rho_00_closest", "calc_cylindrical_radius(Knerec, bhabha_IP_changed)")
    .Define("rho_pm_closest", "calc_cylindrical_radius(KchrecClosest, bhabha_IP)")
    .Define("rho_omega", "hypot(rho_00_closest, rho_pm_closest)")
    .Define("z_00_closest", "Knerec[8] - KchrecClosest[8]")
    .Define("z_pm_closest", "KchrecClosest[8] - Bz")
    ###############################################################################################
    ######################### Variables for bad tracks geometrical cut ############################
    .Define("rho_pm_00", "hypot(Kchrec[6] - Knerec[6], Kchrec[7] - Knerec[7])")
    .Define("z_pm_00", "Kchrec[8] - Knerec[8]")
    .Define("cos_trk1_K", "cosine_between_vectors(trk1, Kchrec)")
    .Define("cos_trk2_K", "cosine_between_vectors(trk2, Kchrec)")
    ###############################################################################################
    ######################## Delta t ##############################################################
    .Define("Delta_t_fit", "KaonChTimeCMSignalFit - KaonNeTimeCMSignalFit")
    .Define("Delta_t_MC", "KaonChTimeCMMC - KaonNeTimeCMMC")
    .Define("Delta_t_rec", "KaonChTimeCMBoostLor - KaonNeTimeCMBoostLor")
    ###############################################################################################
    ######################## Resolutions ##########################################################
    .Define("Delta_t_fit_resolution", "Delta_t_fit - Delta_t_MC")
    .Define("Delta_t_rec_resolution", "Delta_t_rec - Delta_t_MC")
    .Define("trk1_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[0]")
    .Define("trk2_matched", "get_matched_vector(trk1, trk2, trk1MC, trk2MC)[1]")
    .Define("trk1_px_resolution", "trk1_matched[0] - trk1MC[0]")
    .Define("trk2_px_resolution", "trk2_matched[0] - trk2MC[0]")
    .Define("trk1_py_resolution", "trk1_matched[1] - trk1MC[1]")
    .Define("trk2_py_resolution", "trk2_matched[1] - trk2MC[1]")
    .Define("trk1_pz_resolution", "trk1_matched[2] - trk1MC[2]")
    .Define("trk2_pz_resolution", "trk2_matched[2] - trk2MC[2]")
    .Define("Kchrec_px_resolution", "Kchrec[0] - Kchmc[0]")
    .Define("Kchrec_py_resolution", "Kchrec[1] - Kchmc[1]")
    .Define("Kchrec_pz_resolution", "Kchrec[2] - Kchmc[2]")
    .Define("Kchrec_E_resolution", "Kchrec[3] - Kchmc[3]")
    .Define("Kchrec_x_resolution", "Kchrec[6] - Kchmc[6]")
    .Define("Kchrec_y_resolution", "Kchrec[7] - Kchmc[7]")
    .Define("Kchrec_z_resolution", "Kchrec[8] - Kchmc[8]")
    .Define("Kchboost_px_resolution", "Kchboost[0] - Kchmc[0]")
    .Define("Kchboost_py_resolution", "Kchboost[1] - Kchmc[1]")
    .Define("Kchboost_pz_resolution", "Kchboost[2] - Kchmc[2]")
    .Define("Kchboost_E_resolution", "Kchboost[3] - Kchmc[3]")
    .Define("Knerec_px_resolution", "Knerec[0] - Knemc[0]")
    .Define("Knerec_py_resolution", "Knerec[1] - Knemc[1]")
    .Define("Knerec_pz_resolution", "Knerec[2] - Knemc[2]")
    .Define("Knerec_E_resolution", "Knerec[3] - Knemc[3]")
    .Define("Knerec_x_resolution", "Knerec[6] - Knemc[6]")
    .Define("Knerec_y_resolution", "Knerec[7] - Knemc[7]")
    .Define("Knerec_z_resolution", "Knerec[8] - Knemc[8]")    
    .Define("KnetriKinFit_px_resolution", "KnetriKinFit[0] - Knemc[0]")
    .Define("KnetriKinFit_py_resolution", "KnetriKinFit[1] - Knemc[1]")
    .Define("KnetriKinFit_pz_resolution", "KnetriKinFit[2] - Knemc[2]")
    .Define("KnetriKinFit_E_resolution", "KnetriKinFit[3] - Knemc[3]")
    .Define("KnetriKinFit_x_resolution", "KnetriKinFit[6] - Knemc[6]")
    .Define("KnetriKinFit_y_resolution", "KnetriKinFit[7] - Knemc[7]")
    .Define("KnetriKinFit_z_resolution", "KnetriKinFit[8] - Knemc[8]")
    .Define("KchrecFit_px_resolution", "KchrecFit[0] - Kchmc[0]")
    .Define("KchrecFit_py_resolution", "KchrecFit[1] - Kchmc[1]")
    .Define("KchrecFit_pz_resolution", "KchrecFit[2] - Kchmc[2]")
    .Define("KchrecFit_E_resolution", "KchrecFit[3] - Kchmc[3]")
    .Define("KchrecFit_x_resolution", "KchrecFit[6] - Kchmc[6]")
    .Define("KchrecFit_y_resolution", "KchrecFit[7] - Kchmc[7]")
    .Define("KchrecFit_z_resolution", "KchrecFit[8] - Kchmc[8]")
    .Define("KchboostFit_px_resolution", "KchboostFit[0] - Kchmc[0]")
    .Define("KchboostFit_py_resolution", "KchboostFit[1] - Kchmc[1]")
    .Define("KchboostFit_pz_resolution", "KchboostFit[2] - Kchmc[2]")
    .Define("KchboostFit_E_resolution", "KchboostFit[3] - Kchmc[3]")
    .Define("KchboostFit_x_resolution", "KchboostFit[6] - Kchmc[6]")
    .Define("KchboostFit_y_resolution", "KchboostFit[7] - Kchmc[7]")
    .Define("KchboostFit_z_resolution", "KchboostFit[8] - Kchmc[8]")       
    .Define("KnerecFit_px_resolution", "KnerecFit[0] - Knemc[0]")
    .Define("KnerecFit_py_resolution", "KnerecFit[1] - Knemc[1]")
    .Define("KnerecFit_pz_resolution", "KnerecFit[2] - Knemc[2]")
    .Define("KnerecFit_E_resolution", "KnerecFit[3] - Knemc[3]")
    .Define("KnerecFit_x_resolution", "KnerecFit[6] - Knemc[6]")
    .Define("KnerecFit_y_resolution", "KnerecFit[7] - Knemc[7]")
    .Define("KnerecFit_z_resolution", "KnerecFit[8] - Knemc[8]")
    .Define("ip_bhabha_x_resolution", "Bx - ipmc[0]")
    .Define("ip_bhabha_y_resolution", "By - ipmc[1]")
    .Define("ip_bhabha_z_resolution", "Bz - ipmc[2]")
    .Define("ip_x_resolution", "ip[0] - ipmc[0]")
    .Define("ip_y_resolution", "ip[1] - ipmc[1]")
    .Define("ip_z_resolution", "ip[2] - ipmc[2]")
    .Define("ipFit_x_resolution", "ipFit[0] - ipmc[0]")
    .Define("ipFit_y_resolution", "ipFit[1] - ipmc[1]")
    .Define("ipFit_z_resolution", "ipFit[2] - ipmc[2]")
    .Define("KchrecPMFit_px_resolution", "KchrecPMFit[0] - Kchmc[0]")
    .Define("KchrecPMFit_py_resolution", "KchrecPMFit[1] - Kchmc[1]")
    .Define("KchrecPMFit_pz_resolution", "KchrecPMFit[2] - Kchmc[2]")
    .Define("KchrecPMFit_E_resolution", "KchrecPMFit[3] - Kchmc[3]")
    .Define("KchrecPMFit_x_resolution", "KchrecPMFit[6] - Kchmc[6]")
    .Define("KchrecPMFit_y_resolution", "KchrecPMFit[7] - Kchmc[7]")
    .Define("KchrecPMFit_z_resolution", "KchrecPMFit[8] - Kchmc[8]")
    .Define("KchboostPMFit_px_resolution", "KchboostPMFit[0] - Kchmc[0]")
    .Define("KchboostPMFit_py_resolution", "KchboostPMFit[1] - Kchmc[1]")
    .Define("KchboostPMFit_pz_resolution", "KchboostPMFit[2] - Kchmc[2]")
    .Define("KchboostPMFit_E_resolution", "KchboostPMFit[3] - Kchmc[3]")
    .Define("KchboostPMFit_x_resolution", "KchboostPMFit[6] - Kchmc[6]")
    .Define("KchboostPMFit_y_resolution", "KchboostPMFit[7] - Kchmc[7]")
    .Define("KchboostPMFit_z_resolution", "KchboostPMFit[8] - Kchmc[8]")
    .Define("KnerecNeutralFit_px_resolution", "KnerecNeutralFit[0] - Knemc[0]")
    .Define("KnerecNeutralFit_py_resolution", "KnerecNeutralFit[1] - Knemc[1]")
    .Define("KnerecNeutralFit_pz_resolution", "KnerecNeutralFit[2] - Knemc[2]")
    .Define("KnerecNeutralFit_E_resolution", "KnerecNeutralFit[3] - Knemc[3]")
    .Define("KnerecNeutralFit_x_resolution", "KnerecNeutralFit[6] - Knemc[6]")
    .Define("KnerecNeutralFit_y_resolution", "KnerecNeutralFit[7] - Knemc[7]")
    .Define("KnerecNeutralFit_z_resolution", "KnerecNeutralFit[8] - Knemc[8]")
    .Define("Kchrec_5_resolution", f"Kchrec[5] - {mK0}")
    .Define("Knerec_5_resolution", f"Knerec[5] - {mK0}")
    .Define("KnerecSix_5_resolution", f"KnerecSix[5] - {mK0}")
    .Define("KnetriKinFit_5_resolution", f"KnetriKinFit[5] - {mK0}")
    .Define("MassPi01_resolution", f"pi01[5] - {mPi0}")
    .Define("MassPi02_resolution", f"pi02[5] - {mPi0}")
    .Define("MassPi01Fit_resolution", f"pi01Fit[5] - {mPi0}")
    .Define("MassPi02Fit_resolution", f"pi02Fit[5] - {mPi0}")
    .Define("MassPi0NeutralFit1_resolution", f"pi0NeutralFit1[5] - {mPi0}")
    .Define("MassPi0NeutralFit2_resolution", f"pi0NeutralFit2[5] - {mPi0}")
    .Define("MassPi0PMFit1_resolution", f"pi0PMFit1[5] - {mPi0}")
    .Define("MassPi0PMFit2_resolution", f"pi0PMFit2[5] - {mPi0}")
    ###############################################################################################   
    .Filter("mcflag == 1 && mctruth != -1")
)

df = (
  df.Define("Kchrec_5", f"Kchrec[5] - {mK0}")
      .Define("Knerec_5", f"Knerec[5] - {mK0}")
      .Define("KnerecSix_5", f"KnerecSix[5] - {mK0}")
      .Define("KnetriKinFit_5", f"KnetriKinFit[5] - {mK0}")
      .Define("MassPi01", f"pi01[5] - {mPi0}")
      .Define("MassPi02", f"pi02[5] - {mPi0}")
      .Define("MassPi01Fit", f"pi01Fit[5] - {mPi0}")
      .Define("MassPi02Fit", f"pi02Fit[5] - {mPi0}")
      .Define("MassPi0NeutralFit1", f"pi0NeutralFit1[5] - {mPi0}")
      .Define("MassPi0NeutralFit2", f"pi0NeutralFit2[5] - {mPi0}")
      .Define("MassPi0PMFit1", f"pi0PMFit1[5] - {mPi0}")
      .Define("MassPi0PMFit2", f"pi0PMFit2[5] - {mPi0}")
)

# Filtrowanie dla próbki semileptonicznej (dziedziczy wszystkie nowe kolumny m_*_hyp)
df_Semileptonic = df.Filter(
    "mcflag == 1 && (mctruth == 5 || mctruth == 0 || mctruth == -1)"
)

# df Signal
df_Signal = df.Filter(
    f"mcflag == 1 && mctruth == 1"
)

df_Background = df.Filter(
    f"mcflag == 1 && mctruth != 1 && mctruth != -1"
)

Przetestowano 351 plików. Poprawnych: 351, Odrzuconych: 0
Przetestowano 420 plików. Poprawnych: 420, Odrzuconych: 0
Lumi factor: 1.0000


In [28]:
base_path = "/workspace/root_notebooks/results"
cache_path = "/workspace/analysis_snapshot"
analysis_dir = "signal_analysis_cut_checks"

current_date = time.strftime("%Y-%m-%d")

metadata = {
    "saved_errors": "Semileptonic",
    "cutset": "NONE",
    "luminosity_pb_MC": lumi_mc,
    "luminosity_pb_data": lumi_data,
    "scale_factor": scaling_factor,
}

analysis_hash = hashlib.md5(json.dumps(metadata, sort_keys=True).encode()).hexdigest()
metadata["analysis_hash"] = analysis_hash

current_result_dir = f"{base_path}/{analysis_dir}/{analysis_hash}"
cache_path = f"{cache_path}/{analysis_dir}/{analysis_hash}"
config_path = f"{base_path}/{analysis_dir}/config"

os.makedirs(current_result_dir, exist_ok=True)
with open(f"{current_result_dir}/analysis_config.json", "w") as f:
  json.dump(metadata, f, indent=4)

########################## LOAD HISTOGRAM CONFIGURATION ##############################
hist_config_file_path = f"{base_path}/{analysis_dir}/config/histograms_config_1D.yaml"
with open(hist_config_file_path, "r") as f:
    hist_config = yaml.safe_load(f)
######################################################################################

############ LOAD VARIABLES RESOLUTIONS PER CUT HASH #################################
resolutions = {}
list_of_dirs = [
    f for f in os.listdir(config_path) if os.path.isdir(os.path.join(config_path, f))
]

for d in list_of_dirs:
  resolution_file_path = f"{config_path}/{d}/resolutions_config.yaml"
  with open(resolution_file_path, "r") as f:
      resolutions_config = yaml.safe_load(f)
  resolutions[d] = resolutions_config
######################################################################################

In [29]:
try:
  variables_1D = list(hist_config.get("histograms_1d", {}).keys())
  TRUTH_ONLY_VARS_1D = hist_config.get("truth_only_vars_1d", [])
  TRUTH_MCTRUTH_ID_1D = hist_config.get("truth_mctruth_id_1d", 1)

  variables_1D_titles = []
  unit_1D = []
  range_X_1D = {}
  modes = {}

  for var in variables_1D:
      variables_1D_titles.append(hist_config["histograms_1d"][var].get("title", ""))
      unit_1D.append(hist_config["histograms_1d"][var].get("unit", ""))
      modes[var] = list(hist_config["histograms_1d"][var].get("range_x", {}).keys())

      range_X_1D[var] = {}
      for mode in modes[var]:
          range_X_1D[var][mode] = hist_config["histograms_1d"][var].get("range_x", {}).get(mode, [])
      
except Exception as e:
  print(f"Error occurred: {e}")

In [ ]:
## Prepare 1D histogram model on levels:
## cut_hash --> mode --> variable

def prepare_hist_model(cut_hash: list, vars: list, vars_titles: list, unit: list, range_X: dict, resolutions: dict) -> dict:
  hist_model = {}

  if isinstance(cut_hash, str):
      cut_hash = [cut_hash]

  for cut in cut_hash:
      hist_model[cut] = {}

      for var in variables_1D:
        for mode in modes[var]:
          if mode not in hist_model[cut]:
              hist_model[cut][mode] = {}

          ## Prepare sigma for the variable
          sigma_core = resolutions.get(cut, {}).get('variables', {}).get(var, {}).get('sigma_core', [1.0])[0]
          sigma_tail = resolutions.get(cut, {}).get('variables', {}).get(var, {}).get('sigma_tail', [1.0])[0]

          ## Calculate the range length for the variable in the given mode
          range_X_min, range_X_max = range_X[var].get(mode, [0, 1])
          range_len = range_X_max - range_X_min

          if "resolution" in var:
            if range_len < 0.5:
              sigma_core = 0.001
              sigma_tail = 0.001
            elif range_len < 3:
              sigma_core = 0.01
              sigma_tail = 0.01
            elif range_len < 50:
              sigma_core = 0.1
              sigma_tail = 0.1
            else:
              sigma_core = 1.0
              sigma_tail = 1.0
          
          sigma = min(sigma_core, sigma_tail)

          title = f"{vars_titles[vars.index(var)]} [{unit[vars.index(var)]}]" if var in vars else ""

          hist_model[cut][mode][var] = {
              "title": title,
              "range_x": range_X[var].get(mode, []),
              "n_bins": int(range_len / sigma) if sigma > 0 else range_len,
              "resolution": sigma
          }

  return hist_model

In [31]:
histo1Dmodel = prepare_hist_model('62d706c5', variables_1D, variables_1D_titles, unit_1D, range_X_1D, resolutions)

print(histo1Dmodel)

{'62d706c5': {'full': {'Knerec_5': {'title': 'm^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]', 'range_x': [-600, 400], 'n_bins': 1000, 'resolution': 1.0}, 'KnerecSix_5': {'title': 'm^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]', 'range_x': [-400, 200], 'n_bins': 600, 'resolution': 1.0}, 'Kchrec_5': {'title': 'm^{inv}_{+-} - m_{K^{0}} [MeV/c^{2}]', 'range_x': [-200, 200], 'n_bins': 400, 'resolution': 1.0}, 'Chi2SignalKinFit': {'title': '#chi^{2}_{+-00} []', 'range_x': [0, 500], 'n_bins': 500, 'resolution': 1.0}, 'Chi2NeutralKinFit': {'title': '#chi^{2}_{00} []', 'range_x': [0, 500], 'n_bins': 500, 'resolution': 1.0}, 'Chi2PMKinFit': {'title': '#chi^{2}_{+-} []', 'range_x': [0, 100], 'n_bins': 100, 'resolution': 1.0}, 'MassPi01Fit': {'title': 'm^{inv,fit,+-00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]', 'range_x': [-400, 400], 'n_bins': 800, 'resolution': 1.0}, 'MassPi02Fit': {'title': 'm^{inv,fit,+-00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]', 'range_x': [-400, 400], 'n_bins': 800, 'resoluti

In [32]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

full_cut_list = ["abs(Kchrec_5_resolution) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5_resolution, KnerecSix_5_resolution)",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"m1_vs_m2_cut(MassPi01Fit_resolution, MassPi01Fit_resolution, {cut1}, {cut2})",
            "Chi2SignalKinFit < 30.0"
            ]

In [33]:
############# FIRST CUT LIST FOR 000 TAG ################
from utils.hist_utils import fill_1D_histogram_simple


# --- GŁÓWNA PĘTLA BUDUJĄCA CIĘCIA KROK PO KROKU ---
# Możesz zacząć od indeksu -1 (czyli stan "beż żadnych cięć" / nocuts)
# lub od i = 0 (jeśli chcesz zacząć od razu od pierwszego cięcia).
for i in range(-1, len(full_cut_list)):

  # Wycinamy podzbiór cięć od początku do bieżącego kroku
  if i == -1:
    current_cut_list = []  # Stan bazowy bez cięć
    step_name = "00_no_cuts"
  else:
    current_cut_list = full_cut_list[: i + 1]
    step_name = f"step_{i+1:02d}"

  print(
      f"\n[INFO] Przetwarzanie kroku: {step_name} (liczba aktywnych cięć:"
      f" {len(current_cut_list)})"
  )

  # Przygotowanie metadanych, hashu i unikalnego katalogu dla tego kroku
  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = (
      "nocuts"
      if not current_cut_list
      else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
  )

  # Możesz opcjonalnie rozbić wyniki na podkatalogi dla każdego kroku, np.:
  # step_result_dir = os.path.join(current_result_dir, step_name)
  step_result_dir = (
      current_result_dir  # Lub zapisywać w jednym miejscu z hashem w nazwie pliku
  )

  total_cut = build_cut_expression(current_cut_list)

  # Rozdzielczości i model histogramu zależą od bieżącego zestawu cięć.
  histo1Dmodel = prepare_hist_model(
      cut_hash,
      variables_1D,
      variables_1D_titles,
      unit_1D,
      range_X_1D,
      resolutions,
  )
  model_for_cut = histo1Dmodel[cut_hash]

  truth_only_vars = set(TRUTH_ONLY_VARS_1D)
  main_indices = [
      index for index, variable in enumerate(variables_1D)
      if variable not in truth_only_vars
  ]
  truth_indices = [
      index for index, variable in enumerate(variables_1D)
      if variable in truth_only_vars
  ]
  vars_1D_main = [variables_1D[index] for index in main_indices]
  titles_1D_main = [variables_1D_titles[index] for index in main_indices]
  vars_1D_truth = [variables_1D[index] for index in truth_indices]
  titles_1D_truth = [variables_1D_titles[index] for index in truth_indices]

  def modes_for_variables(variables):
    return {
        mode: {}
        for mode, variables_by_mode in model_for_cut.items()
        if all(variable in variables_by_mode for variable in variables)
    }

  modes_1D_main = modes_for_variables(vars_1D_main)
  modes_1D_truth = modes_for_variables(vars_1D_truth)

  # Wypełnianie histogramów głównych: Data + kanały MC + suma MC
  hist_1D = fill_1D_histogram_simple(
      df,
      df_data,
      channName,
      histo1Dmodel,
      cut_hash,
      total_cut,
      variables=vars_1D_main,
      scaling_factor=scaling_factor,
      compute_mc_sum=True,
  )

  # Wypełnianie histogramów truth (jeśli istnieją)
  hist_1D_truth = {}
  if vars_1D_truth:
    truth_channels_1D = {
        TRUTH_MCTRUTH_ID_1D: channName[TRUTH_MCTRUTH_ID_1D]
    }
    hist_1D_truth = fill_1D_histogram_simple(
        df,
        df_data,
        truth_channels_1D,
        histo1Dmodel,
        cut_hash,
        total_cut,
        variables=vars_1D_truth,
        compute_mc_sum=False,
    )

  # Rysowanie i zapis do dedykowanych plików ROOT / PDF
  # (zakładam, że Twoja funkcja draw_and_save przyjmuje cut_hash i zapisuje pliki)
  root_file_name = f"signal_analysis_1D_variables.root"
  hist_1D, canvas, legends = draw_and_save_1d_histograms(
      hist_1D,
      vars_1D_main,
      modes_1D_main,
      channColor,
      channTitle,
      step_result_dir,
      cut_hash,
      cuts_json_str,
      current_cut_list,
      root_file_name,
  )

  if vars_1D_truth:
    root_file_name_truth = (
        f"signal_analysis_1D_truth_variables.root"
    )
    hist_1D_truth, canvas_truth, legends_truth = draw_and_save_1d_histograms(
        hist_1D_truth,
        vars_1D_truth,
        modes_1D_truth,
        channColor,
        channTitle,
        step_result_dir,
        cut_hash,
        cuts_json_str,
        current_cut_list,
        root_file_name_truth,
    )

  # Sprzątanie obiektów Canvas z pamięci ROOT-a
  for canva in canvas:
    canva.Close()
  if vars_1D_truth:
    for canva in canvas_truth:
      canva.Close()

print(
    "[INFO] Pętla sekwencyjnego nakładania cięć została pomyślnie zakończona!"
)


[INFO] Przetwarzanie kroku: 00_no_cuts (liczba aktywnych cięć: 0)
[INFO] Zaktualizowano wpis dla hashu nocuts w rejestrze w /workspace/root_notebooks/results/signal_analysis_cut_checks/63d7f8e132bbb429174651e222962411/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu nocuts w rejestrze w /workspace/root_notebooks/results/signal_analysis_cut_checks/63d7f8e132bbb429174651e222962411/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_01 (liczba aktywnych cięć: 1)
[INFO] Zaktualizowano wpis dla hashu 146f726e w rejestrze w /workspace/root_notebooks/results/signal_analysis_cut_checks/63d7f8e132bbb429174651e222962411/analysis_registry.csv
[INFO] Zaktualizowano wpis dla hashu 146f726e w rejestrze w /workspace/root_notebooks/results/signal_analysis_cut_checks/63d7f8e132bbb429174651e222962411/analysis_registry.csv

[INFO] Przetwarzanie kroku: step_02 (liczba aktywnych cięć: 2)
[INFO] Zaktualizowano wpis dla hashu 0fcaef1b w rejestrze w /workspace/root_notebooks/results/signal_anal

### Get 2D histograms for each cut

In [ ]:
variables_2D = [["Knerec_5", "KnerecSix_5"], 
                ["Chi2SignalKinFit", "Chi2NeutralKinFit"], 
                ["Chi2SignalKinFit", "Chi2PMKinFit"],
                ["Chi2PMKinFit", "Chi2NeutralKinFit"],
                ["MassPi01Fit", "MassPi02Fit"],
                ["MassPi01", "MassPi02"],
                ["MassPi0NeutralFit1", "MassPi0NeutralFit2"],
                ["MassPi0PMFit1", "MassPi0PMFit2"],
                ["Delta_t_MC", "Delta_t_fit_resolution"],
                ["Delta_t_MC", "Delta_t_rec_resolution"]]
variables_2D_titles = [
                       ["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{00} [-]"], 
                       ["#chi^{2}_{+-00} [-]", "#chi^{2}_{+-} [-]"],
                       ["#chi^{2}_{+-} [-]", "#chi^{2}_{00} [-]"],
                       ["m^{inv,+-00}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,+-00}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv,00}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,00}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["m^{inv,+-}_{2#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]", "m^{inv,+-}_{2#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]"],
                       ["#Delta t^{MC} [#tau_{S}]", "#Delta t^{fit} - #Delta t^{MC} [#tau_{S}]"],
                       ["#Delta t^{MC} [#tau_{S}]", "#Delta t^{rec} - #Delta t^{MC} [#tau_{S}]"]
                       ]
range_X_2D = [[(100, -400, 200), (100, -400, 200)], 
              [(100, 0, 100), (100, 0, 100)], 
              [(100, 0, 100), (100, 0, 30)],
              [(100, 0, 30), (100, 0, 100)], 
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -200, 300), (100, -200, 300)],
              [(100, -300, 300), (100, -20, 20)],
              [(100, -300, 300), (100, -20, 20)]]

# Zmienne 2D widoczne TYLKO dla jednego, wybranego mctruth - analogicznie do 1D.
# Przynależność do grupy "truth-only" sprawdzana jest po kluczu "x_y" (var_key).
TRUTH_ONLY_VARS_2D = set([("Delta_t_MC", "Delta_t_fit_resolution"), ("Delta_t_MC", "Delta_t_rec_resolution")])
TRUTH_MCTRUTH_ID_2D = 1

In [ ]:
## 2D histograms
hist_2D = {}
modes = {}

# Definicja zakresów dla trybów: (nbinsX, xlow, xhigh, nbinsY, ylow, yhigh)
modes["full"] = [(100, -400, 200, 100, -400, 200), 
                 (100, 0, 100, 100, 0, 100), 
                 (100, 0, 100, 100, 0, 30),
                 (100, 0, 30, 100, 0, 100),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -200, 300, 100, -200, 300),
                 (100, -300, 300, 100, -20, 20),
                 (100, -300, 300, 100, -20, 20)
                 ]
modes["zoom"] = [(100, -100, 100, 100, -100, 100), 
                 (100, 0, 50, 100, 0, 50), 
                 (100, 0, 50, 100, 0, 15),
                  (100, 0, 15, 100, 0, 50),
                 (100, -50, 50, 100, -50, 50),
                 (100, -50, 50, 100, -50, 50),
                (100, -50, 50, 100, -50, 50),
                (100, -50, 50, 100, -50, 50),
                (100, -100, 100, 100, -10, 10),
                (100, -100, 100, 100, -10, 10)]

(vars_2D_main, titles_2D_main, modes_2D_main), (vars_2D_truth, titles_2D_truth, modes_2D_truth) = \
    split_variable_group(
        variables_2D, variables_2D_titles, modes,
        is_truth_only=lambda pair: "_".join(pair) in TRUTH_ONLY_VARS_2D
    )

hist_2D = fill_2d_histograms(
    df, df_data, channName, vars_2D_main, titles_2D_main, modes_2D_main,
    total_cut, scaling_factor, compute_mc_sum=True
)

hist_2D_truth = {}
if vars_2D_truth:
  truth_channels_2D = {TRUTH_MCTRUTH_ID_2D: channName[TRUTH_MCTRUTH_ID_2D]}
  hist_2D_truth = fill_2d_histograms(
      df, df_data, truth_channels_2D, vars_2D_truth, titles_2D_truth, modes_2D_truth,
      total_cut, compute_mc_sum=False
  )

In [ ]:
root_file_name = "separated_kin_fits_2D_variables.root"
hist_2D, canva = draw_and_save_2d_histograms(
    hist_2D, vars_2D_main, modes_2D_main, current_result_dir, cut_hash,
    cuts_json_str, cut_list, root_file_name
)

if vars_2D_truth:
  root_file_name_truth = "separated_kin_fits_2D_truth_variables.root"
  hist_2D_truth, canva_truth = draw_and_save_2d_histograms(
      hist_2D_truth, vars_2D_truth, modes_2D_truth, current_result_dir, cut_hash,
      cuts_json_str, cut_list, root_file_name_truth
  )

for canva in canvas:
  canva.Close()
if vars_1D_truth:
  for canva in canvas_truth:
    canva.Close()

In [ ]:
import pandas as pd
import json
import hashlib

df = pd.read_csv("results/signal_separated_kin_fits/2026-10-22/analysis_registry.csv")

hash_dict = df[["hash", "cuts_description"]]
hash_dict = hash_dict.set_index("hash").to_dict()["cuts_description"]

# Zdefiniuj swoje pełne listy cięć i parametry tak jak dotychczas:
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"

cut1 = 8.086
cut2 = 11.515

# Hash list order
full_cut_list = [
    "abs(Kchrec_5) <= 3 * 0.8090",
    "ellipse_cut_func(Knerec_5, KnerecSix_5)",
    f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
    f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
    f"m1_vs_m2_cut(MassPi01Fit, MassPi02Fit, {cut1}, {cut2})",
    "Chi2SignalKinFit < 30.0",
]

hash_order = []
for i in range(-1, len(full_cut_list)):

  if i == -1:
    current_cut_list = []  # Stan bazowy bez cięć
    step_name = "00_no_cuts"
  else:
    current_cut_list = full_cut_list[: i + 1]
    step_name = f"step_{i+1:02d}"

  cuts_json_str = json.dumps(current_cut_list, indent=2)
  cut_hash = (
        "nocuts"
        if not current_cut_list
        else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]
    )

  hash_order.append(cut_hash)

In [ ]:
import json
import ROOT

# Getting purity, efficiency and number of events for each cut
results_file = ROOT.TFile("results/signal_separated_kin_fits/2026-10-22/separated_kin_fits_1D_variables.root", "READ")

no_cuts_hash = "nocuts"
num_of_events = {}
purity = {}

for hash, cut in hash_dict.items():
    obj_integrals = results_file.Get(f"{hash}/ChannelIntegrals")
    obj_purity = results_file.Get(f"{hash}/Purity")
    if obj_integrals:
      if hasattr(obj_integrals, "GetTitle"):
        num_of_events[hash] = json.loads(obj_integrals.GetTitle())

    if obj_purity:
      if hasattr(obj_purity, "GetVal"):
        purity[hash] = obj_purity.GetVal()

In [ ]:
import numpy as np
import ROOT

x, y = [], []

# Pamiętaj o użyciu innej nazwy zmiennej niż 'hash' (wbudowana funkcja Pythona)
for i, curr_hash in enumerate(hash_order):
  x.append(float(i))
  y.append(float(purity[curr_hash]))

canva = ROOT.TCanvas("canva", "Purity vs Cut Step", 800, 600)
canva.SetLogy(True)  # Skala logarytmiczna
ROOT.gPad.SetGrid(True, True)

# Liczba binów musi odpowiadać liczbie elementów w y
n_bins = len(y)
h_purity = ROOT.TH1D(
    "h_purity", "Czystość w kolejnych krokach cięć;Krok selekcji;Czystość", n_bins, 0, n_bins
)

# Przykładowe etykiety dla osi X (dostosuj do liczby kroków w hash_order)
labels = ["No cuts", "Cut 1", "Cut 2", "Cut 3", "Cut 4", "Cut 5", "Cut 6"]

for i, val in enumerate(y):
  h_purity.SetBinContent(i + 1, val)
  # Dodanie czytelnych opisów do każdego binu (indeksowanie w ROOT od 1)
  if i < len(labels):
    h_purity.GetXaxis().SetBinLabel(i + 1, labels[i])

h_purity.SetStats(0)  # Usuwa stat box
h_purity.SetFillColor(ROOT.kAzure + 4)

h_purity.Draw("HIST")

# Obrócenie etykiet osi X, jeśli nazwy cięć są dłuższe
h_purity.GetXaxis().SetLabelSize(0.035)
canva.SetBottomMargin(0.2)  # Zostawienie miejsca na dole na opisy

canva.Draw()

### Test of cuts for separated kinematic fit

In [4]:
########### Fiducial volume Omega-pi0 #########################
fiducial_volume_omega = "rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0"
###############################################################
########### Fiducial volume tracks back to back ###############
fiducial_volume_close_tracks = "rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45"
###############################################################

cut1 = 8.086
cut2 = 11.515

cut_list = ["abs(Kchrec_5) <= 3 * 0.8090",
            "ellipse_cut_func(Knerec_5, KnerecSix_5)",
            f"({fiducial_volume_omega} && rho_omega > 1.5) || !({fiducial_volume_omega})",
            f"({fiducial_volume_close_tracks} && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !({fiducial_volume_close_tracks})",
            f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {cut1}, {cut2})",
            "Chi2NeutralKinFit < 30.0 && Chi2PMKinFit < 30.0"
            ]

In [5]:
# Wypuszczasz dane po cięciach bazowych do lokalnego pliku raz przed pętlą:
# 1. Bierzemy wszystkie cięcia OPRÓCZ ostatnich dwóch (do optymalizacji)
base_cuts = cut_list[:-2]
print("Cięcia bazowe do cache:", base_cuts)

cuts_json_str = json.dumps(base_cuts, indent=2)
cut_hash = "nocuts" if not base_cuts else hashlib.md5(cuts_json_str.encode("utf-8")).hexdigest()[:8]

total_base_cut = " && ".join([f"({cut})" for cut in base_cuts])

columns_to_save = [
    # Zdefiniowane zmienne korelacji mas
    "Kchrec_5",
    "Knerec_5",
    "KnerecSix_5",
    "KnetriKinFit_5",
    "MassPi01",
    "MassPi02",
    "MassPi01Fit",
    "MassPi02Fit",
    "MassPi0NeutralFit1",
    "MassPi0NeutralFit2",
    "MassPi0PMFit1",
    "MassPi0PMFit2",
    
    # Cylindryczne i współrzędne
    "rho_ch",
    "rho_ne",
    "z_ch",
    "z_ne",
    
    # Omega-pi0 geometrical cut variables
    "bhabha_IP",
    "bhabha_IP_changed",
    "rho_00_closest",
    "rho_pm_closest",
    "rho_omega",
    "z_00_closest",
    "z_pm_closest",
    
    # Bad tracks geometrical cut variables
    "rho_pm_00",
    "z_pm_00",
    "cos_trk1_K",
    "cos_trk2_K",
    
    # Delta t
    "Delta_t_fit",
    "Delta_t_MC",
    "Delta_t_rec",
    
    # Resolutions i matching
    "Delta_t_fit_resolution",
    "Delta_t_rec_resolution",
    "trk1_matched",
    "trk2_matched",
    "trk1_px_resolution",
    "trk2_px_resolution",
    "trk1_py_resolution",
    "trk2_py_resolution",
    "trk1_pz_resolution",
    "trk2_pz_resolution",

    # Mctruth
    "mctruth",

    # Chi2
    "Chi2SignalKinFit",
    "Chi2NeutralKinFit",
    "Chi2PMKinFit"
]

# Konwersja listy Pythona na wektor C++ wymagany przez RDataFrame.Snapshot w PyROOT
col_vec = ROOT.std.vector("string")()
for col in columns_to_save:
    col_vec.push_back(col)

# 3. Filtrujemy i robimy Snapshot
os.makedirs(f"{cache_path}/{cut_hash}", exist_ok=True)

cache_MC_name = f"{cache_path}/{cut_hash}/local_root_cache.root"
cache_data_name = f"{cache_path}/{cut_hash}/local_root_cache_data.root"

if not os.path.exists(cache_MC_name):
  df_base = df.Filter(total_base_cut)
  df_base.Snapshot(
      "h1", cache_MC_name, col_vec
  )

if not os.path.exists(cache_data_name):
  df_base = df_data.Filter(total_base_cut)
  df_base.Snapshot(
      "h1", cache_data_name, col_vec
  )

Cięcia bazowe do cache: ['abs(Kchrec_5) <= 3 * 0.8090', 'ellipse_cut_func(Knerec_5, KnerecSix_5)', '(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0 && rho_omega > 1.5) || !(rho_pm_closest < 2.0 && rho_00_closest < 1.5 && abs(z_pm_closest) < 2.0 && abs(z_00_closest) < 2.0)', '(rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45 && (abs(cos_trk1_K) < 0.8 || abs(cos_trk2_K) < 0.8) ) || !(rho_pm_00 < 2.05 && abs(z_pm_00) < 2.45)']


### Tests of separated cuts

In [6]:
# Loading cached RDataFrames from local root cache files
df_MC_cache = ROOT.RDataFrame("tree_name", cache_MC_name)
df_data_cache = ROOT.RDataFrame("tree_name", cache_data_name)

In [11]:
import numpy as np
import ROOT

# Włączenie wielowątkowości w ROOT (jeśli jeszcze nie jest włączona)
if not ROOT.IsImplicitMTEnabled():
    ROOT.EnableImplicitMT()

signal_optimized_cuts = {"cut1": 12.086, "cut2": 21.515, "cut3": 40.0, "cut4": 5.0}

num_steps = 10
step_size = 0.5

cut_values = {}
for i in range(1, 5):
    central_val = signal_optimized_cuts[f"cut{i}"]
    offsets = np.arange(-num_steps, num_steps + 1) * step_size
    cut_values[f"cut{i}"] = [central_val + off for off in offsets]

# --- KONFIGURACJA METRYKI I CELU ---
# Etapy wstępne (1 i 2a) optymalizują significance, żeby nie odcinać sygnału przedwcześnie.
# Finałowy etap (2b) może dążyć do docelowych wartości fita globalnego.
FINAL_METRIC_MODE = "target_distance"  # Do wyboru: "significance" lub "target_distance"

# Docelowe parametry z fita globalnego (używane w Etapie 2b dla trybu "target_distance")
TARGET_PURITY = 0.72
TARGET_EFFICIENCY = 0.4906

# --- PARAMETRY WYDAJNOŚCI I CZYSTOŚCI DO OSIĄGNIĘCIA ---
MIN_EFFICIENCY = 0.0  
MIN_PURITY = 0.0      

# Pobieramy całkowitą liczbę sygnału przed cięciami (do liczenia wydajności)
print("Obliczam początkową liczbę sygnału (przed cięciami)...")
n_signal_total = df_MC_cache.Filter("mctruth == 1").Count().GetValue()
print(f"Całkowity sygnał początkowy: {n_signal_total}\n")


# ==========================================
# ETAP 1: Optymalizacja KROKU 1 (cut1 i cut2)
# ==========================================
print("--- Rejestracja zadań: Etap 1 (cut1, cut2) ---")

results_step1 = []

for c1 in cut_values["cut1"]:
    for c2 in cut_values["cut2"]:
        cut_string_step1 = f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {c1}, {c2})"
        
        df_MC_filtered = df_MC_cache.Filter(cut_string_step1)
        
        num_mc_signal_ptr = df_MC_filtered.Filter("mctruth == 1").Count()
        num_mc_background_ptr = df_MC_filtered.Filter("mctruth != 1").Count()
        
        results_step1.append({
            "c1": c1,
            "c2": c2,
            "mc_signal_ptr": num_mc_signal_ptr,
            "mc_background_ptr": num_mc_background_ptr
        })

all_pointers_step1 = [item["mc_signal_ptr"] for item in results_step1] + [item["mc_background_ptr"] for item in results_step1]

print(f"Uruchamiam RunGraphs dla {len(results_step1)} wariantów (Etap 1)...")
ROOT.RDF.RunGraphs(all_pointers_step1)
print("Obliczenia dla Etapu 1 zakończone!")

best_cut1 = signal_optimized_cuts["cut1"]
best_cut2 = signal_optimized_cuts["cut2"]
best_metric_val = -999999  # Etap 1 zawsze maksymalizuje significance
best_eff_1, best_pur_1 = 0.0, 0.0
valid_variants_1 = 0

for item in results_step1:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        valid_variants_1 += 1
        current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
        
        if current_metric > best_metric_val:
            best_metric_val = current_metric
            best_cut1 = item["c1"]
            best_cut2 = item["c2"]
            best_eff_1 = efficiency
            best_pur_1 = purity

print(f"Spełniających kryteria w Etapie 1: {valid_variants_1}/{len(results_step1)}")
print(f"Wyznaczone optymalne wartości (Etap 1): cut1 = {best_cut1}, cut2 = {best_cut2}")
print(f"-> Efficiency: {best_eff_1:.4f}, Purity: {best_pur_1:.4f}\n")


# ==========================================
# ETAP 2a: Optymalizacja KROKU 2a (tylko cut3)
# ==========================================
print("--- Rejestracja zadań: Etap 2a (cut3) ---")

results_step2a = []

for c3 in cut_values["cut3"]:
    cuts_to_test_step2a = [
        f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {best_cut1}, {best_cut2})",
        f"Chi2NeutralKinFit < {c3}"
    ]
    current_cuts_string = " && ".join(cuts_to_test_step2a)
    
    df_MC_filtered = df_MC_cache.Filter(current_cuts_string)
    
    num_mc_signal_ptr = df_MC_filtered.Filter("mctruth == 1").Count()
    num_mc_background_ptr = df_MC_filtered.Filter("mctruth != 1").Count()
    
    results_step2a.append({
        "c3": c3,
        "mc_signal_ptr": num_mc_signal_ptr,
        "mc_background_ptr": num_mc_background_ptr
    })

all_pointers_step2a = [item["mc_signal_ptr"] for item in results_step2a] + [item["mc_background_ptr"] for item in results_step2a]

print(f"Uruchamiam RunGraphs dla {len(results_step2a)} wariantów (Etap 2a)...")
ROOT.RDF.RunGraphs(all_pointers_step2a)
print("Obliczenia dla Etapu 2a zakończone!")

best_cut3 = signal_optimized_cuts["cut3"]
best_metric_val = -999999  # Etap 2a również maksymalizuje significance
best_eff_2a, best_pur_2a = 0.0, 0.0
valid_variants_2a = 0

for item in results_step2a:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        valid_variants_2a += 1
        current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
        
        if current_metric > best_metric_val:
            best_metric_val = current_metric
            best_cut3 = item["c3"]
            best_eff_2a = efficiency
            best_pur_2a = purity

print(f"Spełniających kryteria w Etapie 2a: {valid_variants_2a}/{len(results_step2a)}")
print(f"Wyznaczona optymalna wartość (Etap 2a): cut3 = {best_cut3}")
print(f"-> Efficiency: {best_eff_2a:.4f}, Purity: {best_pur_2a:.4f}\n")


# ==========================================
# ETAP 2b: Optymalizacja KROKU 2b (tylko cut4) - FINAŁ
# ==========================================
print("--- Rejestracja zadań: Etap 2b (cut4) ---")

results_step2b = []

for c4 in cut_values["cut4"]:
    cuts_to_test_step2b = [
        f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {best_cut1}, {best_cut2})",
        f"Chi2NeutralKinFit < {best_cut3}",
        f"Chi2PMKinFit < {c4}"
    ]
    current_cuts_string = " && ".join(cuts_to_test_step2b)
    
    df_MC_filtered = df_MC_cache.Filter(current_cuts_string)
    
    num_mc_signal_ptr = df_MC_filtered.Filter("mctruth == 1").Count()
    num_mc_background_ptr = df_MC_filtered.Filter("mctruth != 1").Count()
    
    results_step2b.append({
        "c4": c4,
        "mc_signal_ptr": num_mc_signal_ptr,
        "mc_background_ptr": num_mc_background_ptr
    })

all_pointers_step2b = [item["mc_signal_ptr"] for item in results_step2b] + [item["mc_background_ptr"] for item in results_step2b]

print(f"Uruchamiam RunGraphs dla {len(results_step2b)} wariantów (Etap 2b)...")
ROOT.RDF.RunGraphs(all_pointers_step2b)
print("Obliczenia dla Etapu 2b zakończone!")

best_cut4 = signal_optimized_cuts["cut4"]
# W etapie finałowym stosujemy wybraną strategię (significance lub target_distance)
best_metric_val = -999999 if FINAL_METRIC_MODE == "significance" else float('inf')
best_eff_2b, best_pur_2b = 0.0, 0.0
valid_variants_2b = 0

for item in results_step2b:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        valid_variants_2b += 1
        
        if FINAL_METRIC_MODE == "significance":
            current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
            is_better = current_metric > best_metric_val
        else:  # "target_distance"
            current_metric = (purity - TARGET_PURITY)**2 + (efficiency - TARGET_EFFICIENCY)**2
            is_better = current_metric < best_metric_val
        
        if is_better:
            best_metric_val = current_metric
            best_cut4 = item["c4"]
            best_eff_2b = efficiency
            best_pur_2b = purity

print(f"Spełniających kryteria w Etapie 2b: {valid_variants_2b}/{len(results_step2b)}")
print(f"Wyznaczona optymalna wartość (Etap 2b): cut4 = {best_cut4}")
print(f"-> Efficiency: {best_eff_2b:.4f}, Purity: {best_pur_2b:.4f}\n")

Obliczam początkową liczbę sygnału (przed cięciami)...
Całkowity sygnał początkowy: 282326

--- Rejestracja zadań: Etap 1 (cut1, cut2) ---
Uruchamiam RunGraphs dla 441 wariantów (Etap 1)...
Obliczenia dla Etapu 1 zakończone!
Spełniających kryteria w Etapie 1: 441/441
Wyznaczone optymalne wartości (Etap 1): cut1 = 12.086, cut2 = 22.515
-> Efficiency: 0.2638, Purity: 0.0634

--- Rejestracja zadań: Etap 2a (cut3) ---
Uruchamiam RunGraphs dla 21 wariantów (Etap 2a)...
Obliczenia dla Etapu 2a zakończone!
Spełniających kryteria w Etapie 2a: 21/21
Wyznaczona optymalna wartość (Etap 2a): cut3 = 35.0
-> Efficiency: 0.2199, Purity: 0.1660

--- Rejestracja zadań: Etap 2b (cut4) ---
Uruchamiam RunGraphs dla 21 wariantów (Etap 2b)...
Obliczenia dla Etapu 2b zakończone!
Spełniających kryteria w Etapie 2b: 21/21
Wyznaczona optymalna wartość (Etap 2b): cut4 = 2.5
-> Efficiency: 0.1743, Purity: 0.5207



In [12]:
import numpy as np
import ROOT

# Włączenie wielowątkowości w ROOT (jeśli jeszcze nie jest włączona)
if not ROOT.IsImplicitMTEnabled():
    ROOT.EnableImplicitMT()

signal_optimized_cuts = {"cut1": 12.086, "cut2": 21.515, "cut3": 40.0, "cut4": 5.0}

num_steps = 10
step_size = 0.5

cut_values = {}
for i in range(1, 5):
    central_val = signal_optimized_cuts[f"cut{i}"]
    offsets = np.arange(-num_steps, num_steps + 1) * step_size
    cut_values[f"cut{i}"] = [central_val + off for off in offsets]

# --- KONFIGURACJA METRYKI I CELU ---
# Etap 1 optymalizuje significance, etap 2 może dążyć do docelowych wartości fita globalnego.
FINAL_METRIC_MODE = "target_distance"  # Do wyboru: "significance" lub "target_distance"

# Docelowe parametry z fita globalnego (używane w Etapie 2 dla trybu "target_distance")
TARGET_PURITY = 0.72
TARGET_EFFICIENCY = 0.4906

# --- PARAMETRY WYDAJNOŚCI I CZYSTOŚCI DO OSIĄGNIĘCIA ---
MIN_EFFICIENCY = 0.0  
MIN_PURITY = 0.0      

# Pobieramy całkowitą liczbę sygnału przed cięciami (do liczenia wydajności)
print("Obliczam początkową liczbę sygnału (przed cięciami)...")
n_signal_total = df_MC_cache.Filter("mctruth == 1").Count().GetValue()
print(f"Całkowity sygnał początkowy: {n_signal_total}\n")


# ==========================================
# ETAP 1: Optymalizacja KROKU 1 (cut1 i cut2)
# ==========================================
print("--- Rejestracja zadań: Etap 1 (cut1, cut2) ---")

results_step1 = []

for c1 in cut_values["cut1"]:
    for c2 in cut_values["cut2"]:
        cut_string_step1 = f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {c1}, {c2})"
        
        df_MC_filtered = df_MC_cache.Filter(cut_string_step1)
        
        num_mc_signal_ptr = df_MC_filtered.Filter("mctruth == 1").Count()
        num_mc_background_ptr = df_MC_filtered.Filter("mctruth != 1").Count()
        
        results_step1.append({
            "c1": c1,
            "c2": c2,
            "mc_signal_ptr": num_mc_signal_ptr,
            "mc_background_ptr": num_mc_background_ptr
        })

all_pointers_step1 = [item["mc_signal_ptr"] for item in results_step1] + [item["mc_background_ptr"] for item in results_step1]

print(f"Uruchamiam RunGraphs dla {len(results_step1)} wariantów (Etap 1)...")
ROOT.RDF.RunGraphs(all_pointers_step1)
print("Obliczenia dla Etapu 1 zakończone!")

best_cut1 = signal_optimized_cuts["cut1"]
best_cut2 = signal_optimized_cuts["cut2"]
best_metric_val = -999999  # Etap 1 zawsze maksymalizuje significance
best_eff_1, best_pur_1 = 0.0, 0.0
valid_variants_1 = 0

for item in results_step1:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        valid_variants_1 += 1
        current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
        
        if current_metric > best_metric_val:
            best_metric_val = current_metric
            best_cut1 = item["c1"]
            best_cut2 = item["c2"]
            best_eff_1 = efficiency
            best_pur_1 = purity

print(f"Spełniających kryteria w Etapie 1: {valid_variants_1}/{len(results_step1)}")
print(f"Wyznaczone optymalne wartości (Etap 1): cut1 = {best_cut1}, cut2 = {best_cut2}")
print(f"-> Efficiency: {best_eff_1:.4f}, Purity: {best_pur_1:.4f}\n")


# ==========================================
# ETAP 2: Optymalizacja KROKU 2 (cut3 i cut4 RAZEM) - FINAŁ
# ==========================================
print("--- Rejestracja zadań: Etap 2 (cut3, cut4) ---")

results_step2 = []

for c3 in cut_values["cut3"]:
    for c4 in cut_values["cut4"]:
        cuts_to_test_step2 = [
            f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {best_cut1}, {best_cut2})",
            f"Chi2NeutralKinFit < {c3}",
            f"Chi2PMKinFit < {c4}"
        ]
        current_cuts_string = " && ".join(cuts_to_test_step2)
        
        df_MC_filtered = df_MC_cache.Filter(current_cuts_string)
        
        num_mc_signal_ptr = df_MC_filtered.Filter("mctruth == 1").Count()
        num_mc_background_ptr = df_MC_filtered.Filter("mctruth != 1").Count()
        
        results_step2.append({
            "c3": c3,
            "c4": c4,
            "mc_signal_ptr": num_mc_signal_ptr,
            "mc_background_ptr": num_mc_background_ptr
        })

all_pointers_step2 = [item["mc_signal_ptr"] for item in results_step2] + [item["mc_background_ptr"] for item in results_step2]

print(f"Uruchamiam RunGraphs dla {len(results_step2)} wariantów (Etap 2)...")
ROOT.RDF.RunGraphs(all_pointers_step2)
print("Obliczenia dla Etapu 2 zakończone!")

best_cut3 = signal_optimized_cuts["cut3"]
best_cut4 = signal_optimized_cuts["cut4"]
# W etapie finałowym stosujemy wybraną strategię (significance lub target_distance)
best_metric_val = -999999 if FINAL_METRIC_MODE == "significance" else float('inf')
best_eff_2, best_pur_2 = 0.0, 0.0
valid_variants_2 = 0

for item in results_step2:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        valid_variants_2 += 1
        
        if FINAL_METRIC_MODE == "significance":
            current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
            is_better = current_metric > best_metric_val
        else:  # "target_distance"
            current_metric = (purity - TARGET_PURITY)**2 + (efficiency - TARGET_EFFICIENCY)**2
            is_better = current_metric < best_metric_val
        
        if is_better:
            best_metric_val = current_metric
            best_cut3 = item["c3"]
            best_cut4 = item["c4"]
            best_eff_2 = efficiency
            best_pur_2 = purity

print(f"Spełniających kryteria w Etapie 2: {valid_variants_2}/{len(results_step2)}")
print(f"Wyznaczone optymalne wartości (Etap 2): cut3 = {best_cut3}, cut4 = {best_cut4}")
print(f"-> Efficiency: {best_eff_2:.4f}, Purity: {best_pur_2:.4f}\n")

Obliczam początkową liczbę sygnału (przed cięciami)...
Całkowity sygnał początkowy: 282326

--- Rejestracja zadań: Etap 1 (cut1, cut2) ---
Uruchamiam RunGraphs dla 441 wariantów (Etap 1)...
Obliczenia dla Etapu 1 zakończone!
Spełniających kryteria w Etapie 1: 441/441
Wyznaczone optymalne wartości (Etap 1): cut1 = 12.086, cut2 = 22.515
-> Efficiency: 0.2638, Purity: 0.0634

--- Rejestracja zadań: Etap 2 (cut3, cut4) ---
Uruchamiam RunGraphs dla 441 wariantów (Etap 2)...
Obliczenia dla Etapu 2 zakończone!
Spełniających kryteria w Etapie 2: 441/441
Wyznaczone optymalne wartości (Etap 2): cut3 = 35.0, cut4 = 2.5
-> Efficiency: 0.1743, Purity: 0.5207



In [18]:
import numpy as np
import ROOT

# Włączenie wielowątkowości w ROOT (jeśli jeszcze nie jest włączona)
if not ROOT.IsImplicitMTEnabled():
    ROOT.EnableImplicitMT()

signal_optimized_cuts = {"cut1": 12.086, "cut2": 21.515, "cut3": 20.0, "cut4": 2.0}

# --- KONFIGURACJA METRYKI I CELU ---
FINAL_METRIC_MODE = "target_distance"  # Do wyboru: "significance" lub "target_distance"

TARGET_PURITY = 0.72
TARGET_EFFICIENCY = 0.4906

MIN_EFFICIENCY = 0.0  
MIN_PURITY = 0.0      

# Pobieramy całkowitą liczbę sygnału przed cięciami
print("Obliczam początkową liczbę sygnału (przed cięciami)...")
n_signal_total = df_MC_cache.Filter("mctruth == 1").Count().GetValue()
print(f"Całkowity sygnał początkowy: {n_signal_total}\n")


# =========================================================================
# FUNKCJA POMOCNICZA DO GENEROWANIA SIATKI WOKÓŁ PUNKTU CENTRALNEGO
# =========================================================================
def make_cut_values(base_cuts, num_steps, step_size):
    cut_vals = {}
    for i in range(1, 5):
        central_val = base_cuts[f"cut{i}"]
        offsets = np.arange(-num_steps, num_steps + 1) * step_size
        cut_vals[f"cut{i}"] = [central_val + off for off in offsets]
    return cut_vals


# =========================================================================
# FUNKCJA POMOCNICZA DO TWORZENIA HISTOGRAMÓW 2D I PROJEKCJI
# =========================================================================
def save_detailed_scan_plots(results, var_x_name, var_y_name, x_vals, y_vals, stage_name, output_file):
    """
    Tworzy TH2D i projekcje dla: Metryki, Wydajności (Efficiency) oraz Czystości (Purity).
    """
    x_min, x_max = min(x_vals), max(x_vals)
    y_min, y_max = min(y_vals), max(y_vals)
    n_x, n_y = len(x_vals), len(y_vals)
    
    dx = (x_vals[1] - x_vals[0]) if n_x > 1 else 1.0
    dy = (y_vals[1] - y_vals[0]) if n_y > 1 else 1.0

    # Definicje histogramów do utworzenia
    metrics_to_plot = [
        ("metric", "Wartość metryki"),
        ("efficiency", "Wydajnost (Efficiency)"),
        ("purity", "Czystość (Purity)")
    ]

    output_file.cd()
    
    for key, title_suffix in metrics_to_plot:
        hname = f"h2d_{key}_{stage_name}"
        htitle = f"{title_suffix} ({stage_name});{var_x_name};{var_y_name}"
        h2d = ROOT.TH2D(hname, htitle, n_x, x_min - dx/2, x_max + dx/2,
                                       n_y, y_min - dy/2, y_max + dy/2)

        # Wypełnianie danych
        for item in results:
            val = item.get(key, 0)
            h2d.Fill(item[var_x_name], item[var_y_name], val)

        h2d.Write()

        # Projekcje (rzuty)
        proj_x = h2d.ProjectionX(f"proj_x_{key}_{stage_name}")
        proj_y = h2d.ProjectionY(f"proj_y_{key}_{stage_name}")
        
        proj_x.SetTitle(f"Rzut {var_x_name} - {title_suffix} ({stage_name});{var_x_name};{title_suffix}")
        proj_y.SetTitle(f"Rzut {var_y_name} - {title_suffix} ({stage_name});{var_y_name};{title_suffix}")

        proj_x.Write()
        proj_y.Write()

    print(f"Zapisano pełny zestaw histogramów dla: {stage_name}")


# Tworzymy plik ROOT na wyniki wizualizacji
plots_file = ROOT.TFile("optimization_plots.root", "RECREATE")


# ==========================================
# ETAP 1: Optymalizacja KROKU 1 (cut1 i cut2)
# ==========================================
print("=== ETAP 1: Optymalizacja cut1 i cut2 ===")

# --- FAZA 1A: Zgrubna (Coarse) ---
print("--- Faza 1A: Skan zgrubny ---")
coarse_cuts_v1 = make_cut_values(signal_optimized_cuts, num_steps=10, step_size=0.5)

results_step1_a = []
for c1 in coarse_cuts_v1["cut1"]:
    for c2 in coarse_cuts_v1["cut2"]:
        cut_string_step1 = f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {c1}, {c2})"
        df_MC_filtered = df_MC_cache.Filter(cut_string_step1)
        
        results_step1_a.append({
            "c1": c1, "c2": c2,
            "mc_signal_ptr": df_MC_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_MC_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step1_a] + [item["mc_background_ptr"] for item in results_step1_a])

best_cut1 = signal_optimized_cuts["cut1"]
best_cut2 = signal_optimized_cuts["cut2"]
best_metric_val = -999999

for item in results_step1_a:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    item["efficiency"] = efficiency
    item["purity"] = purity
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
        item["metric"] = current_metric
        if current_metric > best_metric_val:
            best_metric_val = current_metric
            best_cut1, best_cut2 = item["c1"], item["c2"]
    else:
        item["metric"] = 0

# Zapisujemy histogramy dla Etapu 1 (Coarse)
save_detailed_scan_plots(results_step1_a, "c1", "c2", coarse_cuts_v1["cut1"], coarse_cuts_v1["cut2"], "Etap1_Coarse", plots_file)

print(f"Zgrubne optimum (Etap 1): cut1 = {best_cut1}, cut2 = {best_cut2}")

# --- FAZA 1B: Dokładna (Fine) wokół znalezionego minimum ---
print("--- Faza 1B: Skan dokładny ---")
temp_base_cuts = signal_optimized_cuts.copy()
temp_base_cuts["cut1"] = best_cut1
temp_base_cuts["cut2"] = best_cut2
fine_cuts_v1 = make_cut_values(temp_base_cuts, num_steps=5, step_size=0.05)

results_step1_b = []
for c1 in fine_cuts_v1["cut1"]:
    for c2 in fine_cuts_v1["cut2"]:
        cut_string_step1 = f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {c1}, {c2})"
        df_MC_filtered = df_MC_cache.Filter(cut_string_step1)
        
        results_step1_b.append({
            "c1": c1, "c2": c2,
            "mc_signal_ptr": df_MC_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_MC_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step1_b] + [item["mc_background_ptr"] for item in results_step1_b])

best_metric_val = -999999
best_eff_1, best_pur_1 = 0.0, 0.0

for item in results_step1_b:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    item["efficiency"] = efficiency
    item["purity"] = purity
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
        item["metric"] = current_metric
        if current_metric > best_metric_val:
            best_metric_val = current_metric
            best_cut1, best_cut2 = item["c1"], item["c2"]
            best_eff_1, best_pur_1 = efficiency, purity
    else:
        item["metric"] = 0

# Zapisujemy histogramy dla Etapu 1 (Fine)
save_detailed_scan_plots(results_step1_b, "c1", "c2", fine_cuts_v1["cut1"], fine_cuts_v1["cut2"], "Etap1_Fine", plots_file)

print(f"Dokładne optimum (Etap 1): cut1 = {best_cut1}, cut2 = {best_cut2}")
print(f"-> Efficiency: {best_eff_1:.4f}, Purity: {best_pur_1:.4f}\n")


# ==========================================
# ETAP 2: Optymalizacja KROKU 2 (cut3 i cut4 RAZEM) - FINAŁ
# ==========================================
print("=== ETAP 2: Optymalizacja cut3 i cut4 ===")

# --- FAZA 2A: Zgrubna (Coarse) ---
print("--- Faza 2A: Skan zgrubny ---")
temp_base_cuts_2 = signal_optimized_cuts.copy()
temp_base_cuts_2["cut1"] = best_cut1
temp_base_cuts_2["cut2"] = best_cut2
coarse_cuts_v2 = make_cut_values(temp_base_cuts_2, num_steps=10, step_size=0.5)

results_step2_a = []
for c3 in coarse_cuts_v2["cut3"]:
    for c4 in coarse_cuts_v2["cut4"]:
        cuts_to_test_step2 = [
            f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {best_cut1}, {best_cut2})",
            f"Chi2NeutralKinFit < {c3}",
            f"Chi2PMKinFit < {c4}"
        ]
        df_MC_filtered = df_MC_cache.Filter(" && ".join(cuts_to_test_step2))
        
        results_step2_a.append({
            "c3": c3, "c4": c4,
            "mc_signal_ptr": df_MC_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_MC_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step2_a] + [item["mc_background_ptr"] for item in results_step2_a])

best_cut3 = signal_optimized_cuts["cut3"]
best_cut4 = signal_optimized_cuts["cut4"]
best_metric_val = float('inf') if FINAL_METRIC_MODE == "target_distance" else -999999

for item in results_step2_a:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    item["efficiency"] = efficiency
    item["purity"] = purity
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        if FINAL_METRIC_MODE == "significance":
            current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
            is_better = current_metric > best_metric_val
        else:
            current_metric = (purity - TARGET_PURITY)**2 + (efficiency - TARGET_EFFICIENCY)**2
            is_better = current_metric < best_metric_val
            
        item["metric"] = current_metric
        if is_better:
            best_metric_val = current_metric
            best_cut3, best_cut4 = item["c3"], item["c4"]
    else:
        item["metric"] = 999999 if FINAL_METRIC_MODE == "target_distance" else 0

# Zapisujemy histogramy dla Etapu 2 (Coarse)
save_detailed_scan_plots(results_step2_a, "c3", "c4", coarse_cuts_v2["cut3"], coarse_cuts_v2["cut4"], "Etap2_Coarse", plots_file)

print(f"Zgrubne optimum (Etap 2): cut3 = {best_cut3}, cut4 = {best_cut4}")

# --- FAZA 2B: Dokładna (Fine) wokół znalezionego minimum ---
print("--- Faza 2B: Skan dokładny ---")
temp_base_cuts_2["cut3"] = best_cut3
temp_base_cuts_2["cut4"] = best_cut4
fine_cuts_v2 = make_cut_values(temp_base_cuts_2, num_steps=5, step_size=0.05)

results_step2_b = []
for c3 in fine_cuts_v2["cut3"]:
    for c4 in fine_cuts_v2["cut4"]:
        cuts_to_test_step2 = [
            f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {best_cut1}, {best_cut2})",
            f"Chi2NeutralKinFit < {c3}",
            f"Chi2PMKinFit < {c4}"
        ]
        df_MC_filtered = df_MC_cache.Filter(" && ".join(cuts_to_test_step2))
        
        results_step2_b.append({
            "c3": c3, "c4": c4,
            "mc_signal_ptr": df_MC_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_MC_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step2_b] + [item["mc_background_ptr"] for item in results_step2_b])

best_metric_val = float('inf') if FINAL_METRIC_MODE == "target_distance" else -999999
best_eff_2, best_pur_2 = 0.0, 0.0

for item in results_step2_b:
    n_signal = item["mc_signal_ptr"].GetValue()
    n_background = item["mc_background_ptr"].GetValue()
    efficiency = n_signal / n_signal_total if n_signal_total > 0 else 0
    purity = n_signal / (n_signal + n_background) if (n_signal + n_background) > 0 else 0
    
    item["efficiency"] = efficiency
    item["purity"] = purity
    
    if efficiency >= MIN_EFFICIENCY and purity >= MIN_PURITY:
        if FINAL_METRIC_MODE == "significance":
            current_metric = n_signal / np.sqrt(n_background + n_signal) if (n_background + n_signal) > 0 else 0
            is_better = current_metric > best_metric_val
        else:
            current_metric = (purity - TARGET_PURITY)**2 + (efficiency - TARGET_EFFICIENCY)**2
            is_better = current_metric < best_metric_val
            
        item["metric"] = current_metric
        if is_better:
            best_metric_val = current_metric
            best_cut3, best_cut4 = item["c3"], item["c4"]
            best_eff_2, best_pur_2 = efficiency, purity
    else:
        item["metric"] = 999999 if FINAL_METRIC_MODE == "target_distance" else 0

# Zapisujemy histogramy dla Etapu 2 (Fine)
save_detailed_scan_plots(results_step2_b, "c3", "c4", fine_cuts_v2["cut3"], fine_cuts_v2["cut4"], "Etap2_Fine", plots_file)

# Zamykamy plik ROOT
plots_file.Close()
print("Wszystkie histogramy (Coarse i Fine) zapisane w pliku: optimization_plots.root")

print(f"Dokładne optimum (Etap 2): cut3 = {best_cut3}, cut4 = {best_cut4}")
print(f"-> Efficiency: {best_eff_2:.4f}, Purity: {best_pur_2:.4f}\n")

Obliczam początkową liczbę sygnału (przed cięciami)...
Całkowity sygnał początkowy: 282326

=== ETAP 1: Optymalizacja cut1 i cut2 ===
--- Faza 1A: Skan zgrubny ---
Zapisano pełny zestaw histogramów dla: Etap1_Coarse
Zgrubne optimum (Etap 1): cut1 = 12.086, cut2 = 22.515
--- Faza 1B: Skan dokładny ---
Zapisano pełny zestaw histogramów dla: Etap1_Fine
Dokładne optimum (Etap 1): cut1 = 12.136000000000001, cut2 = 22.515
-> Efficiency: 0.2642, Purity: 0.0633

=== ETAP 2: Optymalizacja cut3 i cut4 ===
--- Faza 2A: Skan zgrubny ---
Zapisano pełny zestaw histogramów dla: Etap2_Coarse
Zgrubne optimum (Etap 2): cut3 = 22.0, cut4 = 2.5
--- Faza 2B: Skan dokładny ---
Zapisano pełny zestaw histogramów dla: Etap2_Fine
Wszystkie histogramy (Coarse i Fine) zapisane w pliku: optimization_plots.root
Dokładne optimum (Etap 2): cut3 = 22.15, cut4 = 2.5
-> Efficiency: 0.1673, Purity: 0.5605



In [19]:
import numpy as np
import ROOT

# Włączenie wielowątkowości w ROOT (jeśli jeszcze nie jest włączona)
if not ROOT.IsImplicitMTEnabled():
    ROOT.EnableImplicitMT()

signal_optimized_cuts = {"cut1": 12.086, "cut2": 21.515, "cut3": 20.0, "cut4": 2.0}

MIN_EFFICIENCY = 0.0  
MIN_PURITY = 0.0      

# Pobieramy całkowitą liczbę sygnału przed cięciami
print("Obliczam początkową liczbę sygnału (przed cięciami)...")
n_signal_total = df_MC_cache.Filter("mctruth == 1").Count().GetValue()
print(f"Całkowity sygnał początkowy: {n_signal_total}\n")


# =========================================================================
# FUNKCJE POMOCNICZE
# =========================================================================
def make_cut_values(base_cuts, num_steps, step_size):
    cut_vals = {}
    for i in range(1, 5):
        central_val = base_cuts[f"cut{i}"]
        offsets = np.arange(-num_steps, num_steps + 1) * step_size
        cut_vals[f"cut{i}"] = [central_val + off for off in offsets]
    return cut_vals


def compute_pareto_front(results):
    """
    Wyznacza punkty należące do Frontu Pareto (maksymalizacja Efficiency i Purity).
    """
    valid = [r for r in results if r["efficiency"] >= MIN_EFFICIENCY and r["purity"] >= MIN_PURITY]
    # Sortujemy malejąco po wydajności, a w przypadku remisu po czystości
    valid_sorted = sorted(valid, key=lambda x: (x["efficiency"], x["purity"]), reverse=True)
    
    pareto_front = []
    max_purity = -1.0
    for item in valid_sorted:
        if item["purity"] > max_purity:
            pareto_front.append(item)
            max_purity = item["purity"]
            
    # Odwracamy, żeby szło od niższej wydajności/wyższej czystości do wyższej wydajności/niższej czystości
    return list(reversed(pareto_front))


def save_detailed_scan_plots(results, var_x_name, var_y_name, x_vals, y_vals, stage_name, output_file):
    x_min, x_max = min(x_vals), max(x_vals)
    y_min, y_max = min(y_vals), max(y_vals)
    n_x, n_y = len(x_vals), len(y_vals)
    
    dx = (x_vals[1] - x_vals[0]) if n_x > 1 else 1.0
    dy = (y_vals[1] - y_vals[0]) if n_y > 1 else 1.0

    metrics_to_plot = [
        ("metric", "Wartość metryki"),
        ("efficiency", "Wydajnost (Efficiency)"),
        ("purity", "Czystość (Purity)")
    ]

    output_file.cd()
    for key, title_suffix in metrics_to_plot:
        hname = f"h2d_{key}_{stage_name}"
        htitle = f"{title_suffix} ({stage_name});{var_x_name};{var_y_name}"
        h2d = ROOT.TH2D(hname, htitle, n_x, x_min - dx/2, x_max + dx/2,
                                       n_y, y_min - dy/2, y_max + dy/2)

        for item in results:
            val = item.get(key, 0)
            h2d.Fill(item[var_x_name], item[var_y_name], val)

        h2d.Write()
        h2d.ProjectionX(f"proj_x_{key}_{stage_name}").Write()
        h2d.ProjectionY(f"proj_y_{key}_{stage_name}").Write()


# Tworzymy plik ROOT na wyniki wizualizacji
plots_file = ROOT.TFile("optimization_plots.root", "RECREATE")


# ==========================================
# ETAP 1: Optymalizacja KROKU 1 (cut1 i cut2)
# ==========================================
print("=== ETAP 1: Optymalizacja cut1 i cut2 ===")

# --- FAZA 1A: Zgrubna (Coarse) ---
print("--- Faza 1A: Skan zgrubny ---")
coarse_cuts_v1 = make_cut_values(signal_optimized_cuts, num_steps=10, step_size=0.5)

results_step1_a = []
for c1 in coarse_cuts_v1["cut1"]:
    for c2 in coarse_cuts_v1["cut2"]:
        df_filtered = df_MC_cache.Filter(f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {c1}, {c2})")
        results_step1_a.append({
            "c1": c1, "c2": c2,
            "mc_signal_ptr": df_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step1_a] + [item["mc_background_ptr"] for item in results_step1_a])

best_cut1 = signal_optimized_cuts["cut1"]
best_cut2 = signal_optimized_cuts["cut2"]
best_metric_val = -999999

for item in results_step1_a:
    n_s = item["mc_signal_ptr"].GetValue()
    n_b = item["mc_background_ptr"].GetValue()
    item["efficiency"] = n_s / n_signal_total if n_signal_total > 0 else 0
    item["purity"] = n_s / (n_s + n_b) if (n_s + n_b) > 0 else 0
    
    current_metric = n_s / np.sqrt(n_b + n_s) if (n_b + n_s) > 0 else 0
    item["metric"] = current_metric
    if current_metric > best_metric_val:
        best_metric_val = current_metric
        best_cut1, best_cut2 = item["c1"], item["c2"]

save_detailed_scan_plots(results_step1_a, "c1", "c2", coarse_cuts_v1["cut1"], coarse_cuts_v1["cut2"], "Etap1_Coarse", plots_file)
print(f"Zgrubne optimum (Etap 1): cut1 = {best_cut1}, cut2 = {best_cut2}")

# --- FAZA 1B: Dokładna (Fine) ---
print("--- Faza 1B: Skan dokładny ---")
temp_base_cuts = signal_optimized_cuts.copy()
temp_base_cuts["cut1"], temp_base_cuts["cut2"] = best_cut1, best_cut2
fine_cuts_v1 = make_cut_values(temp_base_cuts, num_steps=5, step_size=0.05)

results_step1_b = []
for c1 in fine_cuts_v1["cut1"]:
    for c2 in fine_cuts_v1["cut2"]:
        df_filtered = df_MC_cache.Filter(f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {c1}, {c2})")
        results_step1_b.append({
            "c1": c1, "c2": c2,
            "mc_signal_ptr": df_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step1_b] + [item["mc_background_ptr"] for item in results_step1_b])

best_metric_val = -999999
for item in results_step1_b:
    n_s = item["mc_signal_ptr"].GetValue()
    n_b = item["mc_background_ptr"].GetValue()
    item["efficiency"] = n_s / n_signal_total if n_signal_total > 0 else 0
    item["purity"] = n_s / (n_s + n_b) if (n_s + n_b) > 0 else 0
    
    current_metric = n_s / np.sqrt(n_b + n_s) if (n_b + n_s) > 0 else 0
    item["metric"] = current_metric
    if current_metric > best_metric_val:
        best_metric_val = current_metric
        best_cut1, best_cut2 = item["c1"], item["c2"]

save_detailed_scan_plots(results_step1_b, "c1", "c2", fine_cuts_v1["cut1"], fine_cuts_v1["cut2"], "Etap1_Fine", plots_file)
print(f"Dokładne optimum (Etap 1): cut1 = {best_cut1}, cut2 = {best_cut2}\n")


# ==========================================
# ETAP 2: Optymalizacja KROKU 2 (cut3 i cut4 RAZEM) - FINAŁ
# ==========================================
print("=== ETAP 2: Optymalizacja cut3 i cut4 ===")

# --- FAZA 2A: Zgrubna (Coarse) ---
print("--- Faza 2A: Skan zgrubny ---")
temp_base_cuts_2 = signal_optimized_cuts.copy()
temp_base_cuts_2["cut1"], temp_base_cuts_2["cut2"] = best_cut1, best_cut2
coarse_cuts_v2 = make_cut_values(temp_base_cuts_2, num_steps=10, step_size=0.5)

results_step2_a = []
for c3 in coarse_cuts_v2["cut3"]:
    for c4 in coarse_cuts_v2["cut4"]:
        cuts_str = f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {best_cut1}, {best_cut2}) && Chi2NeutralKinFit < {c3} && Chi2PMKinFit < {c4}"
        df_filtered = df_MC_cache.Filter(cuts_str)
        results_step2_a.append({
            "c3": c3, "c4": c4,
            "mc_signal_ptr": df_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step2_a] + [item["mc_background_ptr"] for item in results_step2_a])

for item in results_step2_a:
    n_s = item["mc_signal_ptr"].GetValue()
    n_b = item["mc_background_ptr"].GetValue()
    item["efficiency"] = n_s / n_signal_total if n_signal_total > 0 else 0
    item["purity"] = n_s / (n_s + n_b) if (n_s + n_b) > 0 else 0
    item["metric"] = n_s / np.sqrt(n_b + n_s) if (n_b + n_s) > 0 else 0

save_detailed_scan_plots(results_step2_a, "c3", "c4", coarse_cuts_v2["cut3"], coarse_cuts_v2["cut4"], "Etap2_Coarse", plots_file)

# --- FAZA 2B: Dokładna (Fine) ---
print("--- Faza 2B: Skan dokładny ---")
# Wybieramy wstępnie najlepszy cut3 i cut4 z fazy 2a wg significance
best_cut3_a = max(results_step2_a, key=lambda x: x["metric"])["c3"]
best_cut4_a = max(results_step2_a, key=lambda x: x["metric"])["c4"]

temp_base_cuts_2["cut3"], temp_base_cuts_2["cut4"] = best_cut3_a, best_cut4_a
fine_cuts_v2 = make_cut_values(temp_base_cuts_2, num_steps=5, step_size=0.05)

results_step2_b = []
for c3 in fine_cuts_v2["cut3"]:
    for c4 in fine_cuts_v2["cut4"]:
        cuts_str = f"m1_vs_m2_cut(MassPi0NeutralFit1, MassPi0NeutralFit2, {best_cut1}, {best_cut2}) && Chi2NeutralKinFit < {c3} && Chi2PMKinFit < {c4}"
        df_filtered = df_MC_cache.Filter(cuts_str)
        results_step2_b.append({
            "c3": c3, "c4": c4,
            "mc_signal_ptr": df_filtered.Filter("mctruth == 1").Count(),
            "mc_background_ptr": df_filtered.Filter("mctruth != 1").Count()
        })

ROOT.RDF.RunGraphs([item["mc_signal_ptr"] for item in results_step2_b] + [item["mc_background_ptr"] for item in results_step2_b])

for item in results_step2_b:
    n_s = item["mc_signal_ptr"].GetValue()
    n_b = item["mc_background_ptr"].GetValue()
    item["efficiency"] = n_s / n_signal_total if n_signal_total > 0 else 0
    item["purity"] = n_s / (n_s + n_b) if (n_s + n_b) > 0 else 0
    item["metric"] = n_s / np.sqrt(n_b + n_s) if (n_b + n_s) > 0 else 0

save_detailed_scan_plots(results_step2_b, "c3", "c4", fine_cuts_v2["cut3"], fine_cuts_v2["cut4"], "Etap2_Fine", plots_file)


# ==========================================
# WYZNACZANIE I ZAPIS FRONTU PARETO
# ==========================================
print("\n=== WYZNACZANIE FRONTU PARETO (Etap Finałowy) ===")
pareto_front = compute_pareto_front(results_step2_b)

print(f"Znaleziono {len(pareto_front)} punktów na froncie Pareto:")
print(f"{'Wydajność (Eff)':<18} | {'Czystość (Pur)':<18} | {'cut3':<10} | {'cut4':<10}")
print("-" * 62)
for p in pareto_front:
    print(f"{p['efficiency']:<18.4f} | {p['purity']:<18.4f} | {p['c3']:<10.2f} | {p['c4']:<10.2f}")

# Tworzymy TGraph z Frontem Pareto i zapisujemy do pliku ROOT
plots_file.cd()
n_pareto = len(pareto_front)
effs = np.array([p['efficiency'] for p in pareto_front], dtype=float)
purs = np.array([p['purity'] for p in pareto_front], dtype=float)

pareto_graph = ROOT.TGraph(n_pareto, effs, purs)
pareto_graph.SetName("Pareto_Front_Eff_vs_Pur")
pareto_graph.SetTitle("Front Pareto: Czystosc vs Wydajnosc;Wydajnosc (Efficiency);Czystosc (Purity)")
pareto_graph.SetMarkerStyle(20)
pareto_graph.SetMarkerColor(ROOT.kRed)
pareto_graph.Write()

plots_file.Close()
print("\nFront Pareto oraz wszystkie mapy zapisano pomyślnie do pliku: optimization_plots.root")

Obliczam początkową liczbę sygnału (przed cięciami)...
Całkowity sygnał początkowy: 282326

=== ETAP 1: Optymalizacja cut1 i cut2 ===
--- Faza 1A: Skan zgrubny ---
Zgrubne optimum (Etap 1): cut1 = 12.086, cut2 = 22.515
--- Faza 1B: Skan dokładny ---
Dokładne optimum (Etap 1): cut1 = 12.136000000000001, cut2 = 22.515

=== ETAP 2: Optymalizacja cut3 i cut4 ===
--- Faza 2A: Skan zgrubny ---
--- Faza 2B: Skan dokładny ---

=== WYZNACZANIE FRONTU PARETO (Etap Finałowy) ===
Znaleziono 105 punktów na froncie Pareto:
Wydajność (Eff)    | Czystość (Pur)     | cut3       | cut4      
--------------------------------------------------------------
0.1855             | 0.5186             | 24.75      | 3.75      
0.1855             | 0.5185             | 24.80      | 3.75      
0.1856             | 0.5184             | 24.85      | 3.75      
0.1856             | 0.5183             | 24.90      | 3.75      
0.1856             | 0.5181             | 24.95      | 3.75      
0.1857             | 0.518